# 08 Time-based evaluation of the email/web features (step 7, Part A)
Pre-registered in `docs/PROJECT_LOG.md` section 6f (cutoffs, sets, reading rule). Forward chaining: train on calendar weeks <= T, test on weeks > T, same users on both sides; plus a double holdout (unseen users x future weeks) on the repeat-0 folds. RF and XGBoost with the fixed notebook-04 hyperparameters; calendar week is NOT a feature; no tuning.

In [1]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from itgnn.comms_eval import double_holdout_masks, load_full, same_rows_user_split_ap, time_split_masks, user_bootstrap_ap
from itgnn.commslogs import comms_feature_sets
from itgnn.data import load_config, prepare
from itgnn.evaluation import alert_budget, evaluate_scores, make_folds
from itgnn.weekly_features import FEATURE_SETS, drop_warmup
from itgnn.windows import load_attack_windows

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml")
SEED, QS = cfg["seed"], [0.25, 0.5, 1, 2]
LOG = open(os.environ.get("PROBE_LOG", os.devnull), "a")
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 60
pd.options.display.max_rows = 200
T0 = time.time()
def tick(s): print(f"[{s}] {time.time() - T0:.0f}s", file=LOG, flush=True)

d = prepare(cfg)
users, labels = d["users"], d["labels"]
windows = load_attack_windows(cfg["raw_dir"])
ins = pd.read_csv(Path(cfg["raw_dir"]) / "answers" / "insiders.csv")
scen = ins[ins["dataset"].astype(str) == "4.2"].set_index("user")["scenario"]; scen_of = scen.to_dict()
SCEN = sorted(scen.unique())
cached = Path("data/processed/comms_weekly_full.parquet").exists()
full = load_full(d, windows)
weekly = drop_warmup(full).reset_index(drop=True); y = weekly["label"].to_numpy()
saved = pd.read_parquet("results/weekly_v2_oof_scores.parquet", columns=["user", "week_start", "model"])
k2 = saved[saved["model"] == "RF W2"].drop_duplicates(["user", "week_start"])[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True)
pd.testing.assert_frame_equal(weekly[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True), k2)
ALL = comms_feature_sets(FEATURE_SETS["W2"])
SETS = {k: ALL[k] for k in ["W2", "W2+E+H", "W2+E+H generic"]}
assert not any("cal_week" in v for v in SETS.values())
assert np.isfinite(weekly[sorted({c for v in SETS.values() for c in v})].to_numpy(float)).all()
print(f"cache existed: {cached}; rows {len(full):,} -> {len(weekly):,} after warm-up; attack weeks {int(y.sum())}; same row keys as the saved W2 run")
oof25 = pd.read_parquet("results/comms_25fold_oof_scores.parquet")
tick("setup")

cache existed: False; rows 67,238 -> 63,238 after warm-up; attack weeks 322; same row keys as the saved W2 run


## A0. Test/train blocks per cutoff (T = 59 is shown only to document why it is dropped)

In [2]:
rows = []
for T in [29, 39, 49, 59]:
    tr, te = time_split_masks(weekly, T)
    atr, ate = weekly.loc[tr & (y == 1), "user"], weekly.loc[te & (y == 1), "user"]
    r = {"T": T, "train rows": int(tr.sum()), "train attack weeks": int(len(atr)), "train insiders": atr.nunique(),
         "test rows": int(te.sum()), "test attack weeks": int(len(ate)), "test insiders": ate.nunique(),
         "seen insiders (attack weeks on both sides)": len(set(atr) & set(ate)), "test prevalence": len(ate) / te.sum()}
    for s in SCEN: r[f"test insiders s{s}"] = int(ate[ate.isin(scen[scen == s].index)].nunique())
    rows.append(r)
a0 = pd.DataFrame(rows).set_index("T"); display(a0)
CUTS = [T for T in [29, 39, 49] if a0.loc[T, "test attack weeks"] >= 40]
assert CUTS == [29, 39, 49] and a0.loc[59, "test attack weeks"] < 40
print("kept cutoffs:", CUTS, "| dropped: 59 (", int(a0.loc[59, 'test attack weeks']), "test attack weeks)")

,train rows,train attack weeks,train insiders,test rows,test attack weeks,test insiders,seen insiders (attack weeks on both sides),test prevalence,test insiders s1,test insiders s2,test insiders s3
T,,,,,,,,,,,
29,24581,43,17,38657,279,62,9,0.007,26,30,6
39,34074,149,36,29164,173,39,5,0.006,16,19,4
49,43252,228,53,19986,94,21,4,0.005,8,12,1
59,52103,284,68,11135,38,9,7,0.003,1,7,1


kept cutoffs: [29, 39, 49] | dropped: 59 ( 38 test attack weeks)


## A1-A3. Time split, same users on both sides (18 fits)

In [3]:
def make(a, ytr):
    if a == "RF":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(), n_jobs=4, random_state=SEED, verbosity=0)
def fit_pred(a, cols, tr, te):
    return make(a, y[tr]).fit(weekly.loc[tr, cols], y[tr]).predict_proba(weekly.loc[te, cols])[:, 1]
res, oof_t, met = {}, [], []
for T in CUTS:
    tr, te = time_split_masks(weekly, T)
    for sname, cols in SETS.items():
        for a in ["RF", "XGB"]:
            s = fit_pred(a, cols, tr, te); res[(T, a, sname)] = s
            ev = evaluate_scores(y[te], s, k=100); name = f"{a} {sname} | T{T}"
            met.append({"T": T, "algo": a, "set": sname, "pr_auc": ev["pr_auc"], "roc_auc": ev["roc_auc"]})
            oof_t.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(), "cal_week": weekly.loc[te, "cal_week"].to_numpy(),
                                       "repeat": 0, "fold": 0, "model": name, "score": s, "label": y[te]}))
met = pd.DataFrame(met); oof_t = pd.concat(oof_t, ignore_index=True)
oof_t.to_parquet("results/time_split_oof_scores.parquet", index=False)
print("saved results/time_split_oof_scores.parquet", oof_t.shape); tick("A1 fits")

saved results/time_split_oof_scores.parquet (526842, 8)


In [4]:
# reference (same rows, user-split) and retention R; bootstrap intervals by user
ref = {(T, a, s): same_rows_user_split_ap(oof25, f"{a} {s}", T) for T in CUTS for a in ["RF", "XGB"] for s in SETS}
met["user-split same rows"] = [ref[(r.T, r.algo, r.set)] for r in met.itertuples()]
met["R"] = met["pr_auc"] / met["user-split same rows"]
boot = {}
for T in CUTS:
    tr, te = time_split_masks(weekly, T)
    for a in ["RF", "XGB"]:
        b = user_bootstrap_ap(y[te], {s: res[(T, a, s)] for s in SETS}, weekly.loc[te, "user"], n_boot=200, seed=SEED)
        boot[(T, a)] = b
ci = lambda v: f"[{np.percentile(v, 2.5):.3f}, {np.percentile(v, 97.5):.3f}]"
met["PR-AUC 95% user-bootstrap"] = [ci(boot[(r.T, r.algo)][r.set]) for r in met.itertuples()]
display(met.set_index(["T", "algo", "set"]))
rows = []
for T in CUTS:
    for a in ["RF", "XGB"]:
        for hi, lo in [("W2+E+H", "W2"), ("W2+E+H generic", "W2"), ("W2+E+H", "W2+E+H generic")]:
            dd = boot[(T, a)][hi] - boot[(T, a)][lo]
            pt = met.query("T == @T and algo == @a and set == @hi")["pr_auc"].iloc[0] - met.query("T == @T and algo == @a and set == @lo")["pr_auc"].iloc[0]
            rows.append({"T": T, "algo": a, "difference": f"{hi} - {lo}", "point": pt, "95% bootstrap": ci(dd)})
print("paired differences with user-bootstrap intervals (same resamples)"); display(pd.DataFrame(rows).set_index(["T", "algo", "difference"]))
tick("A2 metrics")

pr_auc  roc_auc  user-split same rows     R PR-AUC 95% user-bootstrap
T  algo set                                                                                  
29 RF   W2               0.248    0.896                 0.599 0.413            [0.175, 0.349]
   XGB  W2               0.206    0.879                 0.564 0.365            [0.148, 0.283]
   RF   W2+E+H           0.510    0.966                 0.737 0.692            [0.406, 0.647]
   XGB  W2+E+H           0.525    0.971                 0.879 0.598            [0.419, 0.628]
   RF   W2+E+H generic   0.370    0.920                 0.689 0.537            [0.271, 0.488]
   XGB  W2+E+H generic   0.412    0.952                 0.819 0.503            [0.331, 0.502]
39 RF   W2               0.579    0.986                 0.610 0.949            [0.483, 0.698]
   XGB  W2               0.560    0.992                 0.577 0.970            [0.468, 0.682]
   RF   W2+E+H           0.727    0.990                 0.751 0.968            [0.591, 0.885]
   XGB  W2+E+H           0.778    0.991                 0.911 0.853            [0.654, 0.909]
   RF   W2+E+H generic   0.637    0.958                 0.712 0.894            [0.486, 0.784]
   XGB  W2+E+H generic   0.696    0.986                 0.834 0.835            [0.542, 0.824]
49 RF   W2               0.684    0.997                 0.687 0.996            [0.547, 0.797]
   XGB  W2               0.673    0.996                 0.633 1.063            [0.563, 0.805]
   RF   W2+E+H           0.950    1.000                 0.876 1.085            [0.888, 0.990]
   XGB  W2+E+H           0.938    1.000                 0.966 0.970            [0.857, 0.994]
   RF   W2+E+H generic   0.855    0.999                 0.821 1.042            [0.731, 0.939]
   XGB  W2+E+H generic   0.872    0.999                 0.886 0.985            [0.779, 0.939]

paired differences with user-bootstrap intervals (same resamples)


point    95% bootstrap
T  algo difference                                     
29 RF   W2+E+H - W2              0.262   [0.134, 0.402]
        W2+E+H generic - W2      0.123   [0.021, 0.224]
        W2+E+H - W2+E+H generic  0.140   [0.054, 0.229]
   XGB  W2+E+H - W2              0.319   [0.205, 0.421]
        W2+E+H generic - W2      0.206   [0.110, 0.291]
        W2+E+H - W2+E+H generic  0.113   [0.043, 0.175]
39 RF   W2+E+H - W2              0.148   [0.029, 0.274]
        W2+E+H generic - W2      0.057  [-0.037, 0.142]
        W2+E+H - W2+E+H generic  0.090  [-0.001, 0.198]
   XGB  W2+E+H - W2              0.218   [0.089, 0.346]
        W2+E+H generic - W2      0.136   [0.002, 0.225]
        W2+E+H - W2+E+H generic  0.081  [-0.016, 0.232]
49 RF   W2+E+H - W2              0.266   [0.156, 0.424]
        W2+E+H generic - W2      0.171   [0.072, 0.248]
        W2+E+H - W2+E+H generic  0.096   [0.006, 0.219]
   XGB  W2+E+H - W2              0.265   [0.151, 0.397]
        W2+E+H generic - W2      0.200   [0.074, 0.288]
        W2+E+H - W2+E+H generic  0.065  [-0.022, 0.162]

In [5]:
bud = alert_budget(oof_t, labels, scen_of, QS, SEED)
bud[["algo", "set", "T"]] = bud["model"].str.extract(r"^(\w+) (.+) \| T(\d+)$").values
elig = {T: weekly[time_split_masks(weekly, T)[1] & (y == 1)].groupby("user").size().index for T in CUTS}
for T in CUTS:
    print(f"--- T = {T}: insiders with attack weeks in the test block: {len(elig[T])} (s1 {int((scen[list(elig[T])] == 1).sum())}, s2 {int((scen[list(elig[T])] == 2).sum())}, s3 {int((scen[list(elig[T])] == 3).sum())}); normal users {len(set(weekly['user']) - set(labels))}")
    b = bud[bud["T"] == str(T)].set_index(["algo", "set", "q%"])[["caught", "normal flagged", "caught s1", "caught s2", "caught s3", "weeks flagged"]]
    display(b)
tick("A3 budget")

--- T = 29: insiders with attack weeks in the test block: 62 (s1 26, s2 30, s3 6); normal users 930


caught  normal flagged  caught s1  caught s2  caught s3  weeks flagged
algo set            q%                                                                           
RF   W2             0.250      31               1         17         12          2             97
                    0.500      41               4         23         15          3            194
                    1.000      53              27         26         24          3            387
                    2.000      58              86         26         28          4            774
     W2+E+H generic 0.250      33               4         11         16          6             97
                    0.500      41              15         13         22          6            194
                    1.000      52              46         22         24          6            387
                    2.000      54              96         23         25          6            774
     W2+E+H         0.250      25               6          1         19          5             97
                    0.500      37              12         10         22          5            194
                    1.000      52              25         24         22          6            387
                    2.000      59              79         26         27          6            774
XGB  W2             0.250      28               4         11         15          2             97
                    0.500      40              16         17         20          3            194
                    1.000      45              52         19         23          3            387
                    2.000      56              99         23         29          4            774
     W2+E+H generic 0.250      22               4          1         18          3             97
                    0.500      36              21          6         24          6            194
                    1.000      40              57          9         25          6            387
                    2.000      46             148         11         29          6            774
     W2+E+H         0.250      25               4          0         20          5             97
                    0.500      33              19          4         24          5            194
                    1.000      40              54          7         27          6            387
                    2.000      46             126         11         29          6            774

--- T = 39: insiders with attack weeks in the test block: 39 (s1 16, s2 19, s3 4); normal users 930


caught  normal flagged  caught s1  caught s2  caught s3  weeks flagged
algo set            q%                                                                           
RF   W2             0.250      28               0         12         14          2             73
                    0.500      31               4         14         15          2            146
                    1.000      36              18         16         18          2            292
                    2.000      37              62         16         19          2            584
     W2+E+H generic 0.250      23               1          9         12          2             73
                    0.500      29               3         13         13          3            146
                    1.000      34              21         16         14          4            292
                    2.000      36              73         16         16          4            584
     W2+E+H         0.250      22               0         10         10          2             73
                    0.500      33               4         16         15          2            146
                    1.000      36              21         16         16          4            292
                    2.000      38              52         16         18          4            584
XGB  W2             0.250      26               1          9         15          2             73
                    0.500      31               7         13         16          2            146
                    1.000      36              21         15         19          2            292
                    2.000      37              45         16         19          2            584
     W2+E+H generic 0.250      21               2          5         14          2             73
                    0.500      29               9         10         16          3            146
                    1.000      33              27         12         17          4            292
                    2.000      37              55         15         18          4            584
     W2+E+H         0.250      25               1         15          9          1             73
                    0.500      34               5         16         17          1            146
                    1.000      36              32         16         17          3            292
                    2.000      38              62         16         19          3            584

--- T = 49: insiders with attack weeks in the test block: 21 (s1 8, s2 12, s3 1); normal users 930


caught  normal flagged  caught s1  caught s2  caught s3  weeks flagged
algo set            q%                                                                           
RF   W2             0.250      16               1          6          9          1             50
                    0.500      17               7          6         10          1            100
                    1.000      20              25          8         11          1            200
                    2.000      21              67          8         12          1            400
     W2+E+H generic 0.250      12               0          3          8          1             50
                    0.500      18               1          6         11          1            100
                    1.000      21              37          8         12          1            200
                    2.000      21             108          8         12          1            400
     W2+E+H         0.250      12               0          4          7          1             50
                    0.500      21               2          8         12          1            100
                    1.000      21              25          8         12          1            200
                    2.000      21              73          8         12          1            400
XGB  W2             0.250      14               2          3         10          1             50
                    0.500      19               8          6         12          1            100
                    1.000      19              21          6         12          1            200
                    2.000      21              49          8         12          1            400
     W2+E+H generic 0.250      13               1          2         10          1             50
                    0.500      16               8          5         10          1            100
                    1.000      21              32          8         12          1            200
                    2.000      21              69          8         12          1            400
     W2+E+H         0.250      18               0          8          9          1             50
                    0.500      21               5          8         12          1            100
                    1.000      21              21          8         12          1            200
                    2.000      21              49          8         12          1            400

## A4. Reading rule (pre-registered): XGBoost W2+E+H, R and the lift over W2, at the 3 cutoffs

In [6]:
def verdict(m, boots=None):
    x = m[(m["algo"] == "XGB")]
    R = {T: x.query("T == @T and set == 'W2+E+H'")["R"].iloc[0] for T in CUTS}
    lift = {T: x.query("T == @T and set == 'W2+E+H'")["pr_auc"].iloc[0] - x.query("T == @T and set == 'W2'")["pr_auc"].iloc[0] for T in CUTS}
    bad = sum(v < 0.60 for v in R.values()) >= 2 or sum(v < 0.05 for v in lift.values()) >= 2
    good = sum(v >= 0.85 for v in R.values()) >= 2 and sum(v >= 0.10 for v in lift.values()) >= 2
    return ("DOES NOT HOLD" if bad else "HOLDS UP" if good else "DEGRADES (partly calendar-driven)"), R, lift
v, R, lift = verdict(met)
print("TIME SPLIT verdict:", v); print("  R (XGB W2+E+H):", {k: round(x, 3) for k, x in R.items()}); print("  lift W2+E+H - W2 (XGB):", {k: round(x, 3) for k, x in lift.items()})
print("  RF (descriptive): R", {T: round(met.query("T == @T and algo == 'RF' and set == 'W2+E+H'")['R'].iloc[0], 3) for T in CUTS})
tick("A4")

TIME SPLIT verdict: HOLDS UP
  R (XGB W2+E+H): {29: np.float64(0.598), 39: np.float64(0.853), 49: np.float64(0.97)}
  lift W2+E+H - W2 (XGB): {29: np.float64(0.319), 39: np.float64(0.218), 49: np.float64(0.265)}
  RF (descriptive): R {29: np.float64(0.692), 39: np.float64(0.968), 49: np.float64(1.085)}


## A5. Double holdout (unseen users x future weeks), repeat-0 folds, 90 fits

In [7]:
folds = [f for f in make_folds(users, labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED) if f["repeat"] == 0]
dres, doof, dmet, dcount = {}, [], [], []
for T in CUTS:
    for f in folds:
        tr, te = double_holdout_masks(weekly, f, T)
        dcount.append({"T": T, "fold": f["fold"], "train attack weeks": int((y[tr] == 1).sum()), "test rows": int(te.sum()), "test attack weeks": int((y[te] == 1).sum())})
        for sname, cols in SETS.items():
            for a in ["RF", "XGB"]:
                s = fit_pred(a, cols, tr, te)
                dmet.append({"T": T, "fold": f["fold"], "algo": a, "set": sname, "pr_auc": evaluate_scores(y[te], s, k=100)["pr_auc"]})
                doof.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(), "cal_week": weekly.loc[te, "cal_week"].to_numpy(),
                                          "repeat": 0, "fold": f["fold"], "model": f"{a} {sname} | T{T}", "score": s, "label": y[te]}))
dmet, doof, dcount = pd.DataFrame(dmet), pd.concat(doof, ignore_index=True), pd.DataFrame(dcount)
doof.to_parquet("results/double_holdout_oof_scores.parquet", index=False)
print("per-fold test blocks (attack weeks per fold are small; the POOLED block over the 5 folds has the same attack weeks as the time split):"); display(dcount.pivot_table(index="T", columns="fold", values="test attack weeks"))
pooled = []
for (T, a, s), g in doof.assign(T=lambda x: x["model"].str.extract(r"T(\d+)$")[0].astype(int), algo=lambda x: x["model"].str.split().str[0], set=lambda x: x["model"].str.extract(r"^\w+ (.+) \| T")[0]).groupby(["T", "algo", "set"]):
    ev = evaluate_scores(g["label"], g["score"], k=100)
    pooled.append({"T": T, "algo": a, "set": s, "pooled PR-AUC": ev["pr_auc"], "ROC-AUC": ev["roc_auc"], "mean per-fold PR-AUC": dmet.query("T == @T and algo == @a and set == @s")["pr_auc"].mean(),
                   "test attack weeks (pooled)": int(g["label"].sum()), "user-split same rows": ref[(T, a, s)]})
pooled = pd.DataFrame(pooled); pooled["R"] = pooled["pooled PR-AUC"] / pooled["user-split same rows"]; pooled["pr_auc"] = pooled["pooled PR-AUC"]
display(pooled.set_index(["T", "algo", "set"]).drop(columns="pr_auc"))
assert (pooled["test attack weeks (pooled)"] >= 40).all()
v2, R2, lift2 = verdict(pooled)
print("DOUBLE HOLDOUT verdict (pooled, pass/fail eligible: every pooled test block has >= 40 attack weeks):", v2)
print("  R (XGB W2+E+H):", {k: round(x, 3) for k, x in R2.items()}, "| lift:", {k: round(x, 3) for k, x in lift2.items()})
dbud = alert_budget(doof, labels, scen_of, QS, SEED)
dbud[["algo", "set", "T"]] = dbud["model"].str.extract(r"^(\w+) (.+) \| T(\d+)$").values
print("alert budget, double holdout (repeat 0, folds pooled; caught of the insiders with attack weeks in the pooled test block)"); display(dbud.set_index(["T", "algo", "set", "q%"])[["caught", "normal flagged", "caught s1", "caught s2", "caught s3"]])
tick("A5")

per-fold test blocks (attack weeks per fold are small; the POOLED block over the 5 folds has the same attack weeks as the time split):


fold,0,1,2,3,4
T,,,,,
29,64.000,43.000,61.000,60.000,51.000
39,56.000,22.000,45.000,23.000,27.000
49,33.000,6.000,28.000,3.000,24.000


pooled PR-AUC  ROC-AUC  mean per-fold PR-AUC  test attack weeks (pooled)  user-split same rows     R
T  algo set                                                                                                                 
29 RF   W2                      0.215    0.897                 0.258                         279                 0.599 0.359
        W2+E+H                  0.407    0.961                 0.433                         279                 0.737 0.553
        W2+E+H generic          0.265    0.905                 0.304                         279                 0.689 0.385
   XGB  W2                      0.176    0.868                 0.207                         279                 0.564 0.312
        W2+E+H                  0.391    0.955                 0.399                         279                 0.879 0.444
        W2+E+H generic          0.353    0.939                 0.353                         279                 0.819 0.431
39 RF   W2                      0.546    0.980                 0.543                         173                 0.610 0.895
        W2+E+H                  0.590    0.984                 0.621                         173                 0.751 0.786
        W2+E+H generic          0.552    0.947                 0.558                         173                 0.712 0.776
   XGB  W2                      0.519    0.990                 0.510                         173                 0.577 0.900
        W2+E+H                  0.726    0.990                 0.756                         173                 0.911 0.797
        W2+E+H generic          0.673    0.979                 0.630                         173                 0.834 0.807
49 RF   W2                      0.656    0.996                 0.646                          94                 0.687 0.955
        W2+E+H                  0.786    0.997                 0.748                          94                 0.876 0.897
        W2+E+H generic          0.778    0.985                 0.717                          94                 0.821 0.948
   XGB  W2                      0.615    0.994                 0.723                          94                 0.633 0.972
        W2+E+H                  0.901    0.999                 0.846                          94                 0.966 0.933
        W2+E+H generic          0.829    0.997                 0.753                          94                 0.886 0.935

DOUBLE HOLDOUT verdict (pooled, pass/fail eligible: every pooled test block has >= 40 attack weeks): DEGRADES (partly calendar-driven)
  R (XGB W2+E+H): {29: np.float64(0.444), 39: np.float64(0.797), 49: np.float64(0.933)} | lift: {29: np.float64(0.215), 39: np.float64(0.207), 49: np.float64(0.286)}


alert budget, double holdout (repeat 0, folds pooled; caught of the insiders with attack weeks in the pooled test block)


caught  normal flagged  caught s1  caught s2  caught s3
T  algo set            q%                                                            
29 RF   W2             0.250      30               5         16         12          2
                       0.500      41              15         22         16          3
                       1.000      49              44         26         20          3
                       2.000      56              96         26         26          4
39 RF   W2             0.250      30               3         13         15          2
                       0.500      32              13         15         15          2
                       1.000      35              33         15         18          2
                       2.000      37              57         16         18          3
49 RF   W2             0.250      16               9          6          9          1
                       0.500      17              14          6         10          1
                       1.000      19              32          7         11          1
                       2.000      21              57          8         12          1
29 RF   W2+E+H generic 0.250      27               5          9         13          5
                       0.500      36              21         14         17          5
                       1.000      45              48         19         20          6
                       2.000      52             110         23         23          6
39 RF   W2+E+H generic 0.250      24               5         10         11          3
                       0.500      28              11         11         14          3
                       1.000      31              23         14         14          3
                       2.000      33              50         16         14          3
49 RF   W2+E+H generic 0.250      14               9          5          8          1
                       0.500      16              12          6          9          1
                       1.000      18              34          7         10          1
                       2.000      20              65          8         11          1
29 RF   W2+E+H         0.250      23               8          2         16          5
                       0.500      34              15         10         19          5
                       1.000      43              35         17         20          6
                       2.000      57              73         26         25          6
39 RF   W2+E+H         0.250      22               2         11         10          1
                       0.500      27               6         13         12          2
                       1.000      33              23         16         14          3
                       2.000      34              46         16         15          3
49 RF   W2+E+H         0.250      13               6          6          6          1
                       0.500      17              14          7          9          1
                       1.000      18              29          8          9          1
                       2.000      21              44          8         12          1
29 XGB  W2             0.250      22              14          9         11          2
                       0.500      34              26         16         15          3
                       1.000      46              56         20         23          3
                       2.000      55             110         24         27          4
39 XGB  W2             0.250      29               5         11         16          2
                       0.500      32              15         14         16          2
                       1.000      34              26         15         17          2
                       2.000      36              51         16         18          2
49 XGB  W2             0.250      17               7          5         11